# scitex_stats — Multiple Comparison Correction Quick Start

Running many tests inflates the false-positive rate. `scitex_stats.correct` adjusts p-values across a family of comparisons — FDR (Benjamini-Hochberg), Bonferroni, Holm, and more.

**What this notebook covers**

1. Build a family of pairwise comparison results.
2. Apply FDR (Benjamini-Hochberg) correction.
3. Read off the adjusted p-values and rejection decisions.

Companion notebooks:
- `01_basic_ttest.ipynb` — the unified result dict each entry comes from
- `02_test_recommendation.ipynb` — picking the right test in the first place

In [1]:
import pandas as pd

from scitex_stats import correct

## 1. A family of pairwise comparisons

`correct_fdr` takes a list of result-dicts that each carry a `pvalue` key. The other keys (`var_x`, `var_y`, etc.) are preserved on the output so the corrected and original entries line up.

In [2]:
results = [
    {"pvalue": 0.010, "var_x": "A", "var_y": "B"},
    {"pvalue": 0.040, "var_x": "A", "var_y": "C"},
    {"pvalue": 0.030, "var_x": "A", "var_y": "D"},
    {"pvalue": 0.200, "var_x": "B", "var_y": "C"},
    {"pvalue": 0.005, "var_x": "B", "var_y": "D"},
    {"pvalue": 0.080, "var_x": "C", "var_y": "D"},
]
pd.DataFrame(results)

,pvalue,var_x,var_y
0,0.010,A,B
1,0.040,A,C
2,0.030,A,D
3,0.200,B,C
4,0.005,B,D
5,0.080,C,D


## 2. Apply FDR (Benjamini-Hochberg)

BH controls the **false discovery rate** — the expected proportion of false positives among the rejected hypotheses — at level α. It is the default for exploratory pairwise comparisons.

In [3]:
corrected = correct.correct_fdr(results, alpha=0.05, method="bh", verbose=False)
pd.DataFrame(corrected)

,pvalue,var_x,var_y,pstars,rejected,alternative,alpha,alpha_adjusted,pvalue_adjusted,power,n_samples,n_x,n_y,n_pairs
0,0.010,A,B,*,True,two-sided,0.05,0.016667,0.030,NaN,NaN,NaN,NaN,NaN
1,0.040,A,C,ns,False,two-sided,0.05,0.033333,0.060,NaN,NaN,NaN,NaN,NaN
2,0.030,A,D,ns,False,two-sided,0.05,0.025000,0.060,NaN,NaN,NaN,NaN,NaN
3,0.200,B,C,ns,False,two-sided,0.05,0.050000,0.200,NaN,NaN,NaN,NaN,NaN
4,0.005,B,D,*,True,two-sided,0.05,0.008333,0.030,NaN,NaN,NaN,NaN,NaN
5,0.080,C,D,ns,False,two-sided,0.05,0.041667,0.096,NaN,NaN,NaN,NaN,NaN


## 3. Read the adjusted decisions

Side-by-side comparison of original and adjusted p-values, with the rejection decision at α=0.05.

In [4]:
table = pd.DataFrame(
    {
        "comparison": [f"{o['var_x']} vs {o['var_y']}" for o in results],
        "p (original)": [o["pvalue"] for o in results],
        "p (adjusted)": [c["pvalue_adjusted"] for c in corrected],
        "rejected": [c["rejected"] for c in corrected],
    }
)
table

,comparison,p (original),p (adjusted),rejected
0,A vs B,0.010,0.030,True
1,A vs C,0.040,0.060,False
2,A vs D,0.030,0.060,False
3,B vs C,0.200,0.200,False
4,B vs D,0.005,0.030,True
5,C vs D,0.080,0.096,False


## Where to next

- **`01_basic_ttest.ipynb`** — the unified result dict that feeds these comparisons.
- **`02_test_recommendation.ipynb`** — choosing the per-comparison test before correction.
- `correct.correct_bonferroni`, `correct.correct_holm` — same interface, stricter family-wise control.